<a href="https://colab.research.google.com/github/esrakirbas/msc-thesis-reducing-hallucinations-in-medical-diagnosis/blob/main/experiments/Medical_Grounding_Experiment_001_MedQA_questions_and_corpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

In [ ]:
from huggingface_hub import HfApi
info = HfApi().dataset_info("awinml/medqa") #get current revision number of MedQA (afdb980b627374a7f10a0a11ed68455596c52742)
print(info.sha)

afdb980b627374a7f10a0a11ed68455596c52742


In [ ]:
from datasets import load_dataset
questions = load_dataset("awinml/medqa", "questions", revision="afdb980b627374a7f10a0a11ed68455596c52742")
print(questions)
#to explore the first 20 questions (I will select some diagnosis questions among them. SELECTED indices : [2, 6, 15, 19] )
for i, item in enumerate(questions["test"].select(range(20))):
    print(f"\nQuestion {i+1} (index {i}):")
    print(item["question"])
    print("-" * 60)

README.md:   0%|          | 0.00/8.98k [00:00<?, ?B/s]

questions/train.jsonl: reconstructing file:   0%|          |  0.00B / 16.2MB            

questions/train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/2.02M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 10178
    })
    validation: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1272
    })
    test: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1273
    })
})

Question 1 (index 0):
A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication o

In [ ]:
from datasets import load_dataset
books = load_dataset("awinml/medqa", "corpus", revision="afdb980b627374a7f10a0a11ed68455596c52742")
print(books)
#to explore the text books and find the relevant one's indexes. MANUALLY SELECTED TEXT BOOKS : [8]:InternalMed_Harrison and [17]:Surgery_Schwartz
for i, item in enumerate(books["train"]):
  print(f"\nText Book {i} : {item["title"]}")


DatasetDict({
    train: Dataset({
        features: ['doc_id', 'title', 'source_filename', 'text'],
        num_rows: 18
    })
})

Text Book 0 : Anatomy_Gray

Text Book 1 : Biochemistry_Lippincott

Text Book 2 : Cell_Biology_Alberts

Text Book 3 : First_Aid_Step1

Text Book 4 : First_Aid_Step2

Text Book 5 : Gynecology_Novak

Text Book 6 : Histology_Ross

Text Book 7 : Immunology_Janeway

Text Book 8 : InternalMed_Harrison

Text Book 9 : Neurology_Adams

Text Book 10 : Obstentrics_Williams

Text Book 11 : Pathology_Robbins

Text Book 12 : Pathoma_Husain

Text Book 13 : Pediatrics_Nelson

Text Book 14 : Pharmacology_Katzung

Text Book 15 : Physiology_Levy

Text Book 16 : Psichiatry_DSM-5

Text Book 17 : Surgery_Schwartz


In [ ]:
from datasets import load_dataset
#load manually selected questions
questions = load_dataset("awinml/medqa", "questions", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_question_indices = [2, 6, 15, 19]
selected_questions = questions["test"].select(selected_question_indices)
#load manually selected textbooks
books = load_dataset("awinml/medqa", "corpus", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_textbook_indices = [8, 17]
selected_textbooks = books["train"].select(selected_textbook_indices)
#to explore the structure of the books
print(selected_textbooks.column_names)
for i, book in enumerate(selected_textbooks):
  print("\nBook Title:", book["title"])
  print("Text length:", len(book["text"]))
  print("First 1000 characters:")
  print(book["text"][:1000])

['doc_id', 'title', 'source_filename', 'text']

Book Title: InternalMed_Harrison
Text length: 22312859
First 1000 characters:
xxxviii its related products in new formats. Kim Davis, as Associate Managing We are privileged to have compiled this 19th edition and areEditor, has adeptly ensured that the complex production of this multi-enthusiastic about all that it offers our readers. We learned much in the authored textbook proceeded smoothly and efficiently. Dominik Pucek process of editing Harrison’s and hope that you will find this edition a oversaw the production of the new procedural videos and Priscilla uniquely valuable educational resource. Beer expertly oversaw the production of our extensive DVD content. The Editors Jeffrey Herzich ably served as production manager for this new edition.

Chapter 1 The Practice of Medicine the practice of Medicine The Editors THE PHYSICIAN IN THE TWENTY-FIRST CENTURY No greater opportunity, responsibility, or obligation can fall to the lot of a 

In [ ]:
import re #built in regular exp. lib.
from datasets import load_dataset
#load manually selected questions
questions = load_dataset("awinml/medqa", "questions", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_question_indices = [2, 6, 15, 19]
selected_questions = questions["test"].select(selected_question_indices)
#load manually selected textbooks
books = load_dataset("awinml/medqa", "corpus", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_textbook_indices = [8, 17]
selected_textbooks = books["train"].select(selected_textbook_indices)
book_InternalMed = selected_textbooks[0]
book_Surgery = selected_textbooks[1]
#try to find a related passage as a grounding context for the first question selected_index = 2
text = book_InternalMed["text"]
keyword = "urinary tract obstruction" #for question 2 : "atheroembolism"
matches = list(re.finditer(re.escape(keyword), text, re.IGNORECASE)) #escape means treat special chars as normal text
#to see the matched passages. After investigating I used those positions to get the related paragraphs
#for match in matches: #commented out after finding the candidate grounding paragraph
#    position = match.start()
#    print("Position:", position)
#    print(text[max(0, position-100):position+2000])
grounding_contexts = [None] * len(selected_questions)
#Grounding Context for Question 2
related_paragraph_start_position = text.find("Atheroembolism is another cause of limb ischemia. In this conditi")
if related_paragraph_start_position != -1:
  related_paragraph_end_position = text[related_paragraph_start_position:].find("\n") + related_paragraph_start_position
  if related_paragraph_end_position != -1:
    grounding_contexts[0] = text[related_paragraph_start_position:related_paragraph_end_position]
print("grounding_contexts[0] : ", grounding_contexts[0])
#Grounding Context for Question 6
related_paragraph_start_position = 15163868 #I found this from the above for loop
related_paragraph_end_position = text[related_paragraph_start_position:].find("\n") + related_paragraph_start_position
if related_paragraph_end_position != -1:
  grounding_contexts[1] = text[related_paragraph_start_position:related_paragraph_end_position]
related_paragraph_start_position = 15166096 #There are 3 paragraphs for Question 6. I found them from the above for loop. This is the second paragraph
related_paragraph_end_position = text[related_paragraph_start_position:].find("\n") + related_paragraph_start_position
if related_paragraph_end_position != -1:
  grounding_contexts[1] += "\n" + text[related_paragraph_start_position:related_paragraph_end_position]
related_paragraph_start_position = 15166750 #There are 3 paragraphs for Question 6. I found them from the above for loop. This is the third paragraph.
related_paragraph_end_position = text[related_paragraph_start_position:].find("\n") + related_paragraph_start_position
if related_paragraph_end_position != -1:
  grounding_contexts[1] += "\n" + text[related_paragraph_start_position:related_paragraph_end_position]
print("grounding_contexts[1] : ", grounding_contexts[1])


grounding_contexts[0] :  Atheroembolism is another cause of limb ischemia. In this condition, multiple small deposits of fibrin, platelets, and cholesterol debris embolize from proximal atherosclerotic lesions or aneurysmal sites. Large protruding aortic atheromas are a source of emboli that may lead to limb ischemia, as well as stroke and renal insufficiency. Atheroembolism may occur after intraarterial procedures. Since atheroemboli to limbs tend to lodge in the small vessels of the muscle and skin and may not occlude the large vessels, distal pulses usually remain palpable. Patients complain of acute pain and tenderness at the site of embolization. Digital vascular occlusion may result in ischemia and the “blue toe” syndrome; digital necrosis and gangrene may develop (Fig. 302-2). Localized areas of tenderness, pallor, and livedo reticularis (see below) occur at sites of emboli. Skin or muscle biopsy may demonstrate cholesterol crystals.
grounding_contexts[1] :  urinary tract Obstru